1.Load dataset

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "/multi_platform_social_sentiment_evolution.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print(df.shape)
print("=" * 60)

print(df.head())

DATASET LOADED
(150000, 31)
             post_id   platform            timestamp        date  hour_of_day  \
0  TIK20250419000000     TikTok  2025-04-19 01:56:55  2025-04-19            1   
1  TWI20250419000001    Twitter  2025-04-19 05:34:09  2025-04-19            5   
2  INS20250419000002  Instagram  2025-04-19 06:33:36  2025-04-19            6   
3  INS20250419000003  Instagram  2025-04-19 06:42:16  2025-04-19            6   
4  RED20250419000004     Reddit  2025-04-19 06:46:49  2025-04-19            6   

   day_of_week  is_weekend      user_id  followers  account_age_days  ...  \
0            5           1  user_426711        137               306  ...   
1            5           1  user_221610       1974              2310  ...   
2            5           1    user_7998       6471              1990  ...   
3            5           1  user_313440       1366              2057  ...   
4            5           1   user_23343       1349              1445  ...   

   shares comments vie

2.  FEATURE ENGINEERING

 2.1. TIMESTAMP CONVERSION

In [2]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

2.2. CYCLICAL TIME ENCODING

In [3]:
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour_of_day"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour_of_day"] / 24
)

# Day encoding
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

2.4. CONTENT FLAGS

In [4]:
# Has media or not
df["has_media"] = np.where(
    df["media_type"] == "Text",
    0,
    1
)

# Heavy hashtag usage
df["many_hashtags"] = np.where(
    df["num_hashtags"] >= 5,
    1,
    0
)

# Long content flag
platform_median = df.groupby("platform")["content_length"].transform("median")
df["long_content"] = np.where(df["content_length"] > platform_median, 1, 0)


2.5. SENTIMENT INTENSITY

In [5]:
sentiment_cols = [

    "sentiment_positive",
    "sentiment_negative",
    "sentiment_neutral"
]

for col in sentiment_cols:

    if col in df.columns:

        # Force probabilities into [0,1]
        df[col] = df[col].clip(0, 1)

In [6]:
df["sentiment_intensity"] = df[
    [
        "sentiment_positive",
        "sentiment_negative",
        "sentiment_neutral"
    ]
].max(axis=1)


2.6. TOXIC CONTENT FLAG

In [7]:
df["high_toxicity"] = np.where(
    df["toxicity_score"] >= 70,
    1,
    0
)

2.8 CHECK NEW FEATURES

In [8]:
new_features = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "has_media",
    "many_hashtags",
    "long_content",
    "sentiment_intensity",
    "high_toxicity"
]

print("\nNew engineered features:")
print(new_features)


New engineered features:
['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'has_media', 'many_hashtags', 'long_content', 'sentiment_intensity', 'high_toxicity']


2.9 Dedup postid 

In [9]:
print("Duplicate post_id:", df["post_id"].duplicated().sum())
df = df.drop_duplicates(subset=["post_id"]).reset_index(drop=True)

print("\nMissing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Duplicate post_id: 0

Missing values per column:
Series([], dtype: int64)


3.DEFINE TARGET (POPULARITY LABEL)

In [10]:
platform_dfs = []

platforms = df["platform"].unique()
for platform in platforms:
    # FILTER PLATFORM
    temp_df = df[
        df["platform"] == platform
    ].copy()
    # TOP 20% = POPULAR
    threshold = temp_df[
        "total_engagement"
    ].quantile(0.80)

    temp_df["popularity"] = np.where(
        temp_df["total_engagement"] >= threshold,
        1,
        0
    )
    platform_dfs.append(temp_df)

df = pd.concat(
    platform_dfs,
    ignore_index=True
)
print("=" * 60)
print("POPULARITY DISTRIBUTION")
print("=" * 60)

print(df["popularity"].value_counts())

print("\nRatio:")
print(df["popularity"].value_counts(normalize=True))


POPULARITY DISTRIBUTION
popularity
0    119771
1     30229
Name: count, dtype: int64

Ratio:
popularity
0    0.798473
1    0.201527
Name: proportion, dtype: float64


In [13]:
print(df.columns.tolist())

['post_id', 'platform', 'timestamp', 'date', 'hour_of_day', 'day_of_week', 'is_weekend', 'user_id', 'followers', 'account_age_days', 'verified', 'topic', 'language', 'content_length', 'media_type', 'num_hashtags', 'sentiment_category', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'likes', 'shares', 'comments', 'views', 'total_engagement', 'engagement_rate_per_1k_followers', 'hours_since_post', 'viral_coefficient', 'cross_platform_spread', 'toxicity_score', 'location', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'has_media', 'many_hashtags', 'long_content', 'sentiment_intensity', 'high_toxicity', 'popularity']


5.CREATE OUTPUT FOLDERS

In [11]:
from pathlib import Path
base_dir = Path("processed_data")

graph_dir = base_dir / "graph_data"
ml_dir = base_dir / "ml_data"

graph_dir.mkdir(parents=True, exist_ok=True)
ml_dir.mkdir(parents=True, exist_ok=True)

5.REMOVE LEAKAGE COLUMNS

In [12]:

LEAKAGE_COLS = [
    # raw datetime
    "timestamp",
    "date",

    # direct engagement metrics
    "likes",
    "shares",
    "comments",
    "views",
    "total_engagement",

    # derived engagement metrics
    "engagement_rate_per_1k_followers",
    "viral_coefficient",

    # virality outcome
    "cross_platform_spread",

    # future-time leakage
    "hours_since_post"
]

# -----------------------------------------------------
# REMOVE ONLY EXISTING COLUMNS
# -----------------------------------------------------

existing_leakage_cols = [

    col for col in LEAKAGE_COLS
    if col in df.columns
]

df = df.drop(
    columns=existing_leakage_cols
)

print("=" * 60)
print("LEAKAGE COLUMNS REMOVED")
print("=" * 60)

print(existing_leakage_cols)

print("\nCurrent Shape:")
print(df.shape)

LEAKAGE COLUMNS REMOVED
['timestamp', 'date', 'likes', 'shares', 'comments', 'views', 'total_engagement', 'engagement_rate_per_1k_followers', 'viral_coefficient', 'cross_platform_spread', 'hours_since_post']

Current Shape:
(150000, 30)


In [13]:
stats = df.groupby("platform").agg(
    n_posts=("post_id", "count"),
    popular_ratio=("popularity", "mean"),
    avg_followers=("followers", "median"),
    n_topics=("topic", "nunique"),
    n_languages=("language", "nunique"),
).round(3)
print(stats)
stats.to_csv(base_dir / "dataset_statistics.csv")

           n_posts  popular_ratio  avg_followers  n_topics  n_languages
platform                                                               
Facebook      7527          0.201          810.0        15           10
Instagram    30195          0.200         1095.0        15           10
Reddit       37585          0.204          146.0        15           10
TikTok       11887          0.200         1129.0        15           10
Twitter      44676          0.201          396.0        15           10
YouTube      18130          0.200         1885.5        15           10


6. SAVE GRAPH DATASET SPLIT BY PLATFORM

In [14]:
platform_list = df["platform"].unique()

print(platform_list)

for platform_name in platform_list:
    platform_df = df[
        df["platform"] == platform_name
    ].copy()

    print(f"\nPROCESSING: {platform_name}")
    print("Shape:", platform_df.shape)
    graph_df = platform_df.copy()

    graph_path = graph_dir / f"{platform_name.lower()}_graph.csv"

    graph_df.to_csv(
        graph_path,
        index=False
    )

    print(f"Graph dataset saved: {graph_path}")

['TikTok' 'Twitter' 'Instagram' 'Reddit' 'Facebook' 'YouTube']

PROCESSING: TikTok
Shape: (11887, 30)
Graph dataset saved: processed_data\graph_data\tiktok_graph.csv

PROCESSING: Twitter
Shape: (44676, 30)
Graph dataset saved: processed_data\graph_data\twitter_graph.csv

PROCESSING: Instagram
Shape: (30195, 30)
Graph dataset saved: processed_data\graph_data\instagram_graph.csv

PROCESSING: Reddit
Shape: (37585, 30)
Graph dataset saved: processed_data\graph_data\reddit_graph.csv

PROCESSING: Facebook
Shape: (7527, 30)
Graph dataset saved: processed_data\graph_data\facebook_graph.csv

PROCESSING: YouTube
Shape: (18130, 30)
Graph dataset saved: processed_data\graph_data\youtube_graph.csv


7.SAVE Ml DATASET

In [16]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

SEEDS = [42, 123, 2024, 7, 99]

for platform_name in platform_list:
    print("\n" + "=" * 60)
    print(f"PROCESSING: {platform_name}")
    print("=" * 60)

    platform_df = df[df["platform"] == platform_name].copy()
    print("Original Shape:", platform_df.shape)

    remove_cols = ["timestamp", "date", "platform", "sentiment_category"]
    platform_df = platform_df.drop(columns=[c for c in remove_cols if c in platform_df.columns])
    if "sentiment_neutral" in platform_df.columns:
        platform_df = platform_df.drop(columns=["sentiment_neutral"])

    categorical_cols = [c for c in ["topic", "language", "media_type", "location"] if c in platform_df.columns]

    for seed in SEEDS:
        train_raw, test_raw = train_test_split(
            platform_df, test_size=0.2,
            stratify=platform_df["popularity"], random_state=seed
        )

        for col in categorical_cols:
            cats = train_raw[col].astype("category").cat.categories
            train_raw[col] = pd.Categorical(train_raw[col], categories=cats)
            test_raw[col]  = pd.Categorical(test_raw[col],  categories=cats)

        train_df = pd.get_dummies(train_raw, columns=categorical_cols, drop_first=True, dtype=int)
        test_df  = pd.get_dummies(test_raw,  columns=categorical_cols, drop_first=True, dtype=int)
        test_df  = test_df.reindex(columns=train_df.columns, fill_value=0)

        constant_cols = [c for c in train_df.columns if train_df[c].nunique() <= 1]
        if constant_cols:
            train_df = train_df.drop(columns=constant_cols)
            test_df  = test_df.drop(columns=constant_cols, errors="ignore")
            print(f"  [seed {seed}] Removed constant columns: {constant_cols}")

        train_path = ml_dir / f"{platform_name.lower()}_train_seed{seed}.csv"
        test_path  = ml_dir / f"{platform_name.lower()}_test_seed{seed}.csv"
        train_df.to_csv(train_path, index=False)
        test_df.to_csv(test_path, index=False)

        print(f"  [seed {seed}] Train shape: {train_df.shape} | Test shape: {test_df.shape}")
        print(f"  [seed {seed}] Saved: {train_path.name}, {test_path.name}")


PROCESSING: TikTok
Original Shape: (11887, 30)
  [seed 42] Train shape: (9509, 56) | Test shape: (2378, 56)
  [seed 42] Saved: tiktok_train_seed42.csv, tiktok_test_seed42.csv
  [seed 123] Train shape: (9509, 56) | Test shape: (2378, 56)
  [seed 123] Saved: tiktok_train_seed123.csv, tiktok_test_seed123.csv
  [seed 2024] Train shape: (9509, 56) | Test shape: (2378, 56)
  [seed 2024] Saved: tiktok_train_seed2024.csv, tiktok_test_seed2024.csv
  [seed 7] Train shape: (9509, 56) | Test shape: (2378, 56)
  [seed 7] Saved: tiktok_train_seed7.csv, tiktok_test_seed7.csv
  [seed 99] Train shape: (9509, 56) | Test shape: (2378, 56)
  [seed 99] Saved: tiktok_train_seed99.csv, tiktok_test_seed99.csv

PROCESSING: Twitter
Original Shape: (44676, 30)
  [seed 42] Train shape: (35740, 56) | Test shape: (8936, 56)
  [seed 42] Saved: twitter_train_seed42.csv, twitter_test_seed42.csv
  [seed 123] Train shape: (35740, 56) | Test shape: (8936, 56)
  [seed 123] Saved: twitter_train_seed123.csv, twitter_test_s